# SHIVA-LLM — Objective 02, Stage 4A
## HyperKT-guided response model and **assumption-based** instructional simulator

**Runtime:** Google Colab Python 3 → **CPU (None)**; 12–16 GB RAM recommended.  
**Inputs:** `objective02_stage3_reward_dataset.zip` (required), `objective02_stage3b_action_transition_audit.zip` (required).  
**Output:** `objective02_stage4a_simulator.zip`.

**Scientific boundary:** Historical `logged_action` is a shifted difficulty-quantile proxy, **not** a recorded intervention. The response model is calibrated on observed outcomes without treating those proxies as treatments. Counterfactual action effects and knowledge transitions are **explicit assumptions**; simulated returns do not establish real-world policy effectiveness. We preserve the HyperKT train/val/test split, use train-only fitting, and do not use target correctness as an input feature.

In [ ]:
import os, json, zipfile, io, warnings, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
SEED=42
np.random.seed(SEED)
OUT=Path('/content/stage4a_results') if Path('/content').exists() else Path('stage4a_results')
OUT.mkdir(parents=True,exist_ok=True)
print('Ready; output folder:',OUT)

### 1. Upload both ZIP archives

In [ ]:
try:
    from google.colab import files
    uploaded=files.upload()
    paths={Path(k).name:Path(k) for k in uploaded}
except ImportError:
    paths={p.name:p for p in Path('.').glob('*.zip')}

def find_zip(expected):
    matches=[p for name,p in paths.items() if expected in name.lower()]
    if len(matches)!=1: raise FileNotFoundError(f'Expected one archive containing {expected}, found {[str(x) for x in matches]}')
    return matches[0]
reward_zip=find_zip('stage3_reward_dataset')
audit_zip=find_zip('stage3b_action_transition_audit')
with zipfile.ZipFile(reward_zip) as z:
    df=pd.read_csv(z.open('hyperkt_aligned_all_events.csv'),low_memory=False)
    stage3=json.loads(z.read('stage3_audit.json'))
with zipfile.ZipFile(audit_zip) as z:
    stage3b=json.loads(z.read('stage3b_audit.json'))
    adjacency=pd.read_csv(z.open('event_transition_diagnostics.csv'))
assert stage3b['action_is_authenticated_intervention'] is False
assert stage3b['offline_rl_policy_evaluation_ready'] is False
print('Rows:',len(df),'splits:',df.split.value_counts().to_dict())
print('Stage 3B verified adjacency:',stage3b['raw_adjacent_exported_transitions'])

### 2. Data integrity, split separation and pre-response features

In [ ]:
required=['split','user_id','order_id','hyperkt_pred_correct','target_correct','prior_accuracy','prior_attempts','target_skill']
missing=set(required)-set(df.columns)
assert not missing, f'Missing columns: {missing}'
assert set(df.split.unique())=={'train','val','test'}
for a,bb in [('train','val'),('train','test'),('val','test')]:
    assert not (set(df.loc[df.split==a,'user_id']) & set(df.loc[df.split==bb,'user_id']))
assert not df.duplicated(['user_id','order_id']).any()
assert df.hyperkt_pred_correct.between(0,1).all()
assert df.target_correct.isin([0,1]).all()
# Exclude target response, future-state features, split_stage1, and logged_action.
# Prior history and pre-response HyperKT prediction are the only features used.
features=['hyperkt_pred_correct','prior_accuracy','prior_attempts']
for col in features:
    df[col]=pd.to_numeric(df[col],errors='coerce')
train=df.loc[df.split=='train'].copy()
fill={c:float(train[c].median()) for c in features}
for c in features: df[c]=df[c].fillna(fill[c])
assert np.isfinite(df[features].to_numpy()).all()
assert df[features].ge(0).all().all()
print('Feature set:',features,'Train-only medians:',fill)
print('No target outcome, logged action, or future feature enters the response model.')

### 3. Train-only response model and held-out calibration

In [ ]:
Xtrain=df.loc[df.split=='train',features].to_numpy(dtype=float)
ytrain=df.loc[df.split=='train','target_correct'].to_numpy(dtype=int)
model=Pipeline([('scale',StandardScaler()),('logit',LogisticRegression(max_iter=500,random_state=SEED))])
model.fit(Xtrain,ytrain)
records=[]; predictions=[]
for split in ['train','val','test']:
    sub=df.loc[df.split==split]
    p=np.clip(model.predict_proba(sub[features].to_numpy(dtype=float))[:,1],1e-6,1-1e-6)
    y=sub.target_correct.to_numpy(dtype=int)
    metrics={'split':split,'n':len(y),'learners':sub.user_id.nunique(),
             'brier':float(brier_score_loss(y,p)),
             'log_loss':float(log_loss(y,p,labels=[0,1])),
             'auc':float(roc_auc_score(y,p)) if len(np.unique(y))>1 else None,
             'accuracy_0_5':float(accuracy_score(y,p>=0.5)),
             'observed_rate':float(y.mean()),'predicted_rate':float(p.mean())}
    records.append(metrics)
    predictions.append(pd.DataFrame({'split':split,'user_id':sub.user_id.to_numpy(),
      'order_id':sub.order_id.to_numpy(),'observed_correct':y,'p_observed':p}))
metrics=pd.DataFrame(records)
print(metrics.to_string(index=False))
metrics.to_csv(OUT/'response_model_metrics.csv',index=False)
pd.concat(predictions,ignore_index=True).to_csv(OUT/'response_model_predictions.csv',index=False)
joblib.dump({'model':model,'features':features,'fill_values':fill,'seed':SEED},OUT/'response_model.joblib')

In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
for split in ['val','test']:
    sub=pd.concat(predictions,ignore_index=True)
    sub=sub[sub.split==split]
    frac,mean=calibration_curve(sub.observed_correct,sub.p_observed,n_bins=10,strategy='quantile')
    ax.plot(mean,frac,marker='o',label=split)
ax.plot([0,1],[0,1],'--',label='ideal')
ax.set(xlabel='Mean predicted probability',ylabel='Observed correctness rate',title='Held-out observational calibration')
ax.legend();fig.tight_layout();fig.savefig(OUT/'calibration.png',dpi=160);plt.show()

### 4. Explicitly simulated action-response model

Action 0 = easy; 1 = medium; 2 = hard. Historical data **do not identify** their causal effects. For sensitivity scenarios, we assume easy actions increase immediate correctness relative to medium, and hard actions decrease it, by a log-odds offset. This is an imposed counterfactual response rule, **not an estimated treatment effect**. The baseline fitted probability is calibrated to observed responses; its action-specific counterfactual interpretation is not validated.

In [ ]:
from scipy.special import expit, logit
ACTIONS={0:'easy',1:'medium',2:'hard'}
SCENARIOS={'small':0.15,'moderate':0.40,'large':0.80}
def action_prob(base_p,action,effect=0.40):
    if action not in ACTIONS: raise ValueError('action must be 0,1,2')
    base_p=np.clip(np.asarray(base_p,dtype=float),1e-5,1-1e-5)
    return expit(logit(base_p)+(1-action)*effect)

# Reward is simulated and action-specific; do not use observed target_correct as counterfactual reward.
def expected_reward(base_p,action,effect=0.40,weights=(0.6,0.2,0.2)):
    p=action_prob(base_p,action,effect)
    # Assumed efficiency cost: easy 0.15, medium 0.35, hard 0.60
    cost=np.array([0.15,0.35,0.60])[action]
    efficiency=1-cost
    suitability=np.clip(1-abs(p-0.65)/0.65,0,1)
    return weights[0]*p+weights[1]*efficiency+weights[2]*suitability

# No fabricated longitudinal knowledge-gain model: this simulator is a one-step contextual bandit.
# Sequential RL requires separately justified transition dynamics.
for name,effect in SCENARIOS.items():
    print(name,'example P(correct) for baseline 0.60:',
          [round(float(action_prob(0.6,a,effect)),3) for a in ACTIONS])

### 5. Simulator smoke tests, action support and sensitivity results

In [ ]:
assert np.allclose(action_prob(np.array([0.2,0.5,0.8]),1),[0.2,0.5,0.8])
for e in SCENARIOS.values():
    assert np.all(action_prob([0.2,0.5,0.8],0,e)>=action_prob([0.2,0.5,0.8],1,e))
    assert np.all(action_prob([0.2,0.5,0.8],1,e)>=action_prob([0.2,0.5,0.8],2,e))

pred=pd.concat(predictions,ignore_index=True)
# Fixed seed common evaluation contexts, test split only for descriptive sensitivity; no parameter tuning on test.
contexts=pred.loc[pred.split=='test','p_observed'].to_numpy()
rng=np.random.default_rng(SEED)
policies={'fixed_easy':np.zeros(len(contexts),dtype=int),
          'fixed_medium':np.ones(len(contexts),dtype=int),
          'fixed_hard':np.full(len(contexts),2,dtype=int),
          'random':rng.integers(0,3,size=len(contexts))}
rows=[]
for scenario,effect in SCENARIOS.items():
    for name,actions in policies.items():
        # Expected reward: no Monte Carlo outcome noise; explicitly simulated.
        rewards=np.array([expected_reward(contexts[actions==a],a,effect) for a in ACTIONS],dtype=object)
        total=sum(float(np.sum(v)) for v in rewards)
        rows.append({'scenario':scenario,'assumed_log_odds_effect':effect,
                     'policy':name,'n_contexts':len(contexts),'mean_simulated_expected_reward':total/len(contexts)})
sensitivity=pd.DataFrame(rows)
sensitivity.to_csv(OUT/'simulation_sensitivity.csv',index=False)
print(sensitivity.to_string(index=False))
print('NOTE: These are assumption-based expected rewards, NOT measured policy improvements.')

### 6. Save audit, simulator specification and ZIP

In [ ]:
spec={'stage':'4A','method':'one_step_contextual_bandit_simulator','seed':SEED,
      'data_provenance':'HyperKT stage3, stage3b audits',
      'action_labels':ACTIONS,'action_effect_log_odds_scenarios':SCENARIOS,
      'assumed_action_costs':{'easy':0.15,'medium':0.35,'hard':0.60},
      'reward_weights':{'correctness':0.6,'efficiency_proxy':0.2,'suitability_proxy':0.2},
      'feature_columns':features,'train_only_fit':True,
      'split_policy':'original HyperKT learner-disjoint partitions',
      'observed_action_is_authentic':False,'counterfactual_effect_identified':False,
      'longitudinal_transition_model_validated':False,
      'offline_rl_ready':False,
      'note':'Only observational correctness calibration is empirically evaluated; action effects are assumed and cannot support real-world causal claims.'}
(OUT/'simulator_spec.json').write_text(json.dumps(spec,indent=2))
(OUT/'README.txt').write_text('SHIVA-LLM Objective 02 Stage 4A. Empirical response-model calibration + assumption-based one-step contextual bandit simulator. No causal effects or sequential transition dynamics are identified. Do not claim RL policy gains from these outputs.\n')
archive=Path('/content/objective02_stage4a_simulator.zip') if Path('/content').exists() else Path('objective02_stage4a_simulator.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUT.iterdir():z.write(p,p.name)
print('Saved:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass